# Sarcomere Repair Pathway

This notebook models how sarcomere proteins (titin, myosin, actin, troponin) reassemble after microtear damage from resistance training.  
It visualises the assembly graph, estimates a repair timeline, and shows how therapy parameters (leucine dose, sleep quality, temperature) accelerate recovery.

**Pipeline:**
1. Build the assembly graph from `configs/proteins.yaml`
2. Score each domain node for structural stability
3. Simulate repair timeline (baseline vs. optimised therapy)
4. Identify bottleneck domains that slow sarcomere reconstruction

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import math
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

from src.pathway.assembly_graph import (
    build_assembly_graph,
    assembly_step_nodes,
    node_stability_score,
    bottleneck_nodes,
)
from src.pathway.repair_simulator import simulate_repair, optimal_therapy_summary

print('Imports OK')

## 1  Build and inspect the assembly graph

In [ ]:
G = build_assembly_graph()

print(f'Nodes : {G.number_of_nodes()}')
print(f'Edges : {G.number_of_edges()}')
print()
print('Nodes and folding rates:')
for node, data in sorted(G.nodes(data=True), key=lambda x: x[1].get('step', 99)):
    print(f"  [{data['step']}] {node:<45}  folding_rate={data['folding_rate']:.2f}  type={data.get('domain_type', '?')}")

## 2  Visualise the assembly graph

In [ ]:
# Colour nodes by assembly step
step_colours = {1: '#4C72B0', 2: '#55A868', 3: '#C44E52', 4: '#8172B2'}
node_colours = [step_colours.get(G.nodes[n].get('step', 1), '#777') for n in G.nodes]

# Node size proportional to folding_rate (higher = more stable = larger)
node_sizes = [800 * G.nodes[n].get('folding_rate', 0.5) for n in G.nodes]

# Short labels for readability
labels = {n: n.split('__')[1].replace('_', '\n') for n in G.nodes}

fig, ax = plt.subplots(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=2.5)
nx.draw_networkx(
    G, pos=pos, ax=ax,
    labels=labels,
    node_color=node_colours,
    node_size=node_sizes,
    font_size=7,
    edge_color='#aaa',
    arrows=True,
    arrowsize=12,
)

legend_patches = [
    mpatches.Patch(color=c, label=f'Step {s}')
    for s, c in step_colours.items()
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)
ax.set_title('Sarcomere Assembly Graph — nodes sized by folding rate', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3  Stability scores and bottlenecks

In [ ]:
scores = {n: node_stability_score(G, n) for n in G.nodes}
sorted_scores = sorted(scores.items(), key=lambda x: x[1])

nodes_list = [n.split('__')[1] for n, _ in sorted_scores]
score_vals = [s for _, s in sorted_scores]
bar_colours = ['#C44E52' if s < 0.55 else '#4C72B0' for s in score_vals]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(nodes_list, score_vals, color=bar_colours)
ax.axvline(0.55, color='#C44E52', linestyle='--', linewidth=1.2, label='Bottleneck threshold')
ax.set_xlabel('Stability score')
ax.set_title('Domain stability scores (red = bottleneck, score < 0.55)')
ax.legend()
plt.tight_layout()
plt.show()

print('\nBottom 3 bottleneck domains:')
for node_id, score in bottleneck_nodes(G, top_n=3):
    data = G.nodes[node_id]
    print(f'  {node_id:<45}  score={score:.3f}  function="{data.get("function", "?")}')

## 4  Repair timeline: baseline vs. optimised therapy

In [ ]:
baseline = simulate_repair()

optimised = simulate_repair(therapy_params={
    'leucine_dose_g': 5.0,    # 5g leucine post-workout
    'sleep_quality': 0.9,     # high-quality sleep
    'body_temperature_c': 37.2,
})

print(f'Baseline total repair time  : {baseline.total_repair_time_h:.1f} h')
print(f'Optimised total repair time : {optimised.total_repair_time_h:.1f} h')
print(f'Time saved                  : {baseline.total_repair_time_h - optimised.total_repair_time_h:.1f} h')
print(f'Repair score (optimised)    : {optimised.total_repair_score:.3f}')

In [ ]:
# Side-by-side per-step duration bars
step_names = [s.step_name for s in baseline.steps]
baseline_durations = [s.estimated_duration_h for s in baseline.steps]
optimised_durations = [s.estimated_duration_h for s in optimised.steps]

x = range(len(step_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i - width/2 for i in x], baseline_durations, width, label='Baseline', color='#C44E52', alpha=0.85)
ax.bar([i + width/2 for i in x], optimised_durations, width, label='Optimised therapy', color='#55A868', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels([n.replace('_', ' ') for n in step_names], rotation=15, ha='right')
ax.set_ylabel('Estimated duration (h)')
ax.set_title('Sarcomere repair timeline: baseline vs. optimised therapy')
ax.legend()
plt.tight_layout()
plt.show()

## 5  Leucine dose sensitivity analysis

In [ ]:
leucine_doses = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
repair_times = []

for dose in leucine_doses:
    r = simulate_repair(therapy_params={'leucine_dose_g': dose, 'sleep_quality': 0.7})
    repair_times.append(r.total_repair_time_h)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(leucine_doses, repair_times, marker='o', color='#4C72B0', linewidth=2)
ax.set_xlabel('Leucine dose (g)')
ax.set_ylabel('Total repair time (h)')
ax.set_title('Effect of leucine supplementation on sarcomere repair time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_dose = leucine_doses[repair_times.index(min(repair_times))]
print(f'Optimal leucine dose: {best_dose}g  →  repair time {min(repair_times):.1f}h')

## 6  Full repair report

In [ ]:
print(optimal_therapy_summary(optimised))